In [2]:
from pymatgen.core.structure import Structure
from pymatgen.io.vasp import Poscar
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.transformations.standard_transformations import RotationTransformation
from pymatgen.io.ase         import AseAtomsAdaptor
from ase.io.vasp             import write_vasp
import numpy as np
import os
import json

In [3]:
# Define name of folder and path to reference POSCAR
general_folder = 'input/CeO2-heterostructure'

# Step 1: Read the POSCAR files and best terminations for each one
substrate_miller = (3, 1, 1)
substrate_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CONTCAR-0_1_2_i_76"
substrate_structure = Structure.from_file(substrate_POSCAR)  # Load the first surface slab

film_miller = (4, 4, 3)
film_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CONTCAR-0_1_1_i_14"
film_structure = Structure.from_file(film_POSCAR)  # Load the second surface slab

/Users/cibran/work/UPC/SlabOptimization/.venv/lib/python3.13/site-packages/pymatgen/core/structure.py:3087: EncodingWarning: We strongly encourage explicit `encoding`, and we would use UTF-8 by default as per PEP 686
  with zopen(filename, mode="rt", errors="replace") as file:


In [ ]:
from skopt import gp_minimize

def energy_function(angle, distance):

    # Stack with a 30° rotation and 5Å vacuum
    heterostructure = stack(slab1, slab2, axis=2, rotate=30, vacuum=5.0)

    save heterostructure

    energy = sol.read_energy(path to heterostructure)

    return energy

res = gp_minimize(
    energy_function,
    dimensions=[(0, 30), (2.0, 6.0)],  # Angle and distance ranges
    n_calls=50,
    random_state=42
)
print(f"Best angle: {res.x[0]}°, distance: {res.x[1]}Å")

In [4]:
if not os.path.exists(general_folder):
    os.system(f'mkdir {general_folder}')

heterostructure_data = {
        'substrate_miller': substrate_miller,
        'film_miller': film_miller,
        'gap': 2, # Gap between film and substrate
        'vacuum_over_film': 20, # Vacuum over the top of the film
        'film_thickness': 1, # Film thickness
        'substrate_thickness': 1, # Substrate thickness
        'in_layers': True # Set the thickness in layer units
    }

with open(f'{general_folder}/heterostructure_data.json', 'w') as json_file:
    json.dump(heterostructure_data, json_file)

# Copy POSCARs there
os.system(f'cp {substrate_POSCAR} {general_folder}/POSCAR-substrate')
os.system(f'cp {film_POSCAR}      {general_folder}/POSCAR-film')

0

In [5]:
# Step 2: Rotate slab_2 around the c-axis (z-axis)
twist_angle = 15  # Rotation angle in degrees
rotation = RotationTransformation(axis=[0, 0, 1], angle=twist_angle)  # Rotate around z-axis
film_structure = rotation.apply_transformation(film_structure)

In [6]:
# Step 2: Initialize CoherentInterfaceBuilder
interface_builder = CoherentInterfaceBuilder(
    substrate_structure=substrate_structure,  # First slab (substrate)
    film_structure=film_structure,       # Second slab (film)
    substrate_miller=substrate_miller,  # Miller index of the substrate surface
    film_miller=film_miller       # Miller index of the film surface
)

In [7]:
for termination in interface_builder._terminations.keys():
    print(f"\nTrying termination: {termination}")
    interfaces = list(interface_builder.get_interfaces(
        termination=termination,
        gap=2,
        vacuum_over_film=20,
        film_thickness=1,  # Try increasing this
        substrate_thickness=1,  # Try increasing this
        in_layers=True
    ))
    print(f"Generated {len(interfaces)} interfaces for {termination}")


Trying termination: ('BiSBr_P1_192', 'Bi_P-1_2')
Generated 0 interfaces for ('BiSBr_P1_192', 'Bi_P-1_2')


In [8]:
termination

('BiSBr_P1_192', 'Bi_P-1_2')

In [36]:
valid_terminations = list(interface_builder._terminations.keys())

In [37]:
# Step 3: Generate possible interfaces
for t_idx, termination in enumerate(valid_terminations):
    termination_folder = f'{general_folder}/termination-{t_idx}'
    if not os.path.exists(termination_folder):
        os.system(f'mkdir {termination_folder}')

    # Generate heterostructures with given termination
    interfaces = interface_builder.get_interfaces(
        termination=termination,  # Or 'bottom', should compare them
        gap=2, # Gap between film and substrate
        vacuum_over_film=20, # Vacuum over the top of the film
        film_thickness=1, # Film thickness
        substrate_thickness=1, # Substrate thickness
        in_layers=True # Set the thickness in layer units
    )

    for idx, interface in enumerate(list(interfaces)):
        idx_folder = f'{termination_folder}/{idx}'
        if not os.path.exists(idx_folder):
            os.system(f'mkdir {idx_folder}')

        # Save slab structure into miller_folder
        write_vasp(f'{idx_folder}/POSCAR', AseAtomsAdaptor.get_atoms(interface), direct=True, sort=True)

In [28]:
idx_folder

NameError: name 'idx_folder' is not defined